In [ ]:
import torch
import matplotlib.pyplot as plt

from nnAudio.features import STFT, CQT
from src.utils import load_audio, get_freqs, get_freqs_mask, get_spectrum, get_low_hull_curve

DEVICE = torch.device("cpu")

N_FFT = 1 << 14
SR = 44100
BINS_PER_OCTAVE = 192
AREA = 20
FREQ_RANGE = [300, 10000]
FMIN = 32.7

hop_length = N_FFT // 2
fmax = SR / 2  # Maximum frequency that can be represented

freqs = get_freqs(n_fft=N_FFT, sr=SR, log=False, bins_per_octave=BINS_PER_OCTAVE, fmin=FMIN)
mask = get_freqs_mask(freqs, sr=SR, freq_range=FREQ_RANGE)
freqs = freqs.to(DEVICE)
freqs_crop = freqs[mask]
print(len(freqs))

log_freqs = get_freqs(n_fft=N_FFT, sr=SR, log=True, bins_per_octave=BINS_PER_OCTAVE, fmin=FMIN)
log_mask = get_freqs_mask(log_freqs, sr=SR, freq_range=FREQ_RANGE)
log_freqs = log_freqs.to(DEVICE)
log_freqs_crop = log_freqs[log_mask]
print(len(log_freqs))

stft_transform = STFT(
    n_fft=N_FFT,
    sr=SR,
    hop_length=hop_length,
    fmin=FMIN,
    fmax=fmax,
    output_format="Magnitude",
    verbose=False,
).to(DEVICE)

cqt_transform = CQT(
    sr=SR,
    hop_length=hop_length,
    fmin=FMIN,
    fmax=fmax,
    bins_per_octave=BINS_PER_OCTAVE,
    output_format="Magnitude",
    verbose=False,
).to(DEVICE)

In [ ]:
import glob
import soxr

MAX_DURATION = 60.0

data_dir = "/Users/emiledugelay/datasets/fma_small/"
file_paths = glob.glob(f"{data_dir}/**/*.mp3", recursive=True)
audio_paths = file_paths[:4]

waveforms = []
for path in audio_paths:
    waveform, sr = load_audio(path, max_duration=MAX_DURATION)
    if sr != SR:
        print(f"Resampling {path} from {sr} Hz to {SR} Hz")
        waveform = soxr.resample(waveform.T, sr, SR, quality="VHQ").T
        waveform = torch.from_numpy(waveform)
    waveform = waveform.mean(dim=0, keepdim=True)
    waveforms.append(waveform)

lengths = torch.tensor([w.shape[-1] for w in waveforms], device=DEVICE)
L_max = max(w.shape[-1] for w in waveforms)

padded = torch.zeros(len(waveforms), 1, L_max)
for k, w in enumerate(waveforms):
    padded[k, :, :w.shape[-1]] = w
padded = padded.to(DEVICE) # (B, 1, Lmax)

stft_batch = get_spectrum(stft_transform, padded) # (B, n_bins, T')
cqt_batch = get_spectrum(cqt_transform, padded) # (B, n_bins, T')

T_frames = stft_batch.shape[-1]
frame_lengths = ((lengths - N_FFT) // hop_length + 1).unsqueeze(1) # (B, 1)
mask = torch.arange(T_frames, device=DEVICE).unsqueeze(0) < frame_lengths # (B, T')

# Average over time dimension, accounting for varying lengths
stft_batch = (stft_batch * mask.unsqueeze(1)).sum(-1) / frame_lengths.float() # (B, n_bins)
cqt_batch = (cqt_batch * mask.unsqueeze(1)).sum(-1) / frame_lengths.float() # (B, n_bins)


for i in range(stft_batch.shape[0]):
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(freqs.cpu().numpy(), stft_batch[i].cpu().numpy(), label="STFT")
    plt.title("STFT")
    plt.xscale("log")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(log_freqs.cpu().numpy(), cqt_batch[i].cpu().numpy(), label="CQT")
    plt.title("CQT")
    plt.xscale("log")
    plt.legend()

    plt.tight_layout()

In [ ]:
stft_low_hull_curve = get_low_hull_curve(stft_batch, area=AREA)
residue_stft = torch.clamp(stft_batch - stft_low_hull_curve, min=0)

cqt_low_hull_curve = get_low_hull_curve(cqt_batch, area=AREA)
residue_cqt = torch.clamp(cqt_batch - cqt_low_hull_curve, min=0)


for i in range(len(audio_paths)):
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(freqs.cpu().numpy(), stft_batch[i].cpu().numpy())
    plt.plot(freqs.cpu().numpy(), stft_low_hull_curve[i].cpu().numpy(), label="Lower Hull", color="orange")
    plt.title(f"STFT {i+1}")
    plt.xscale("log")

    plt.subplot(1, 2, 2)
    plt.plot(log_freqs.cpu().numpy(), cqt_batch[i].cpu().numpy())
    plt.plot(log_freqs.cpu().numpy(), cqt_low_hull_curve[i].cpu().numpy(), label="Lower Hull", color="orange")
    plt.title(f"CQT {i+1}")
    plt.xscale("log")

In [ ]:
for i in range(len(audio_paths)):
    plt.figure(figsize=(18, 12))
    plt.subplot(1, 2, 1)
    plt.plot(freqs.cpu().numpy(), residue_stft[i].cpu().numpy())
    plt.title(f"STFT Residue {i+1}")
    plt.xscale("log")

    plt.subplot(1, 2, 2)
    plt.plot(log_freqs.cpu().numpy(), residue_cqt[i].cpu().numpy())
    plt.title(f"CQT Residue {i+1}")
    plt.xscale("log")